# Feature engineering — Silver → feature set (Gold)

Builds the model-ready feature table from Silver and saves it as
`features_basic.parquet`, keyed by `wine_id`. Downstream modelling
notebooks load this instead of re-deriving features.

**Scope (leakage-safe only).** This notebook holds deterministic
transforms that do *not* peek at the target or the train/test split:
ordinal encoding (a fixed label map), `log_retail`, and pass-through
numerics. Target encoding and standardisation are intentionally **not**
here — they must be fit on the training split inside the model notebook,
otherwise test information leaks into the features.

In [1]:
import numpy as np
import pandas as pd
import itables
from itables import show
from sklearn.preprocessing import OrdinalEncoder

itables.options.columnDefs = [{"className": "dt-left", "targets": "_all"}]

SILVER_PATH   = r"..\..\.data\wine_reviews_silver.parquet"
FEATURES_PATH = r"..\..\features\features_basic.parquet"

df = pd.read_parquet(SILVER_PATH)
print(f"Silver shape: {df.shape}")
assert df["wine_id"].is_unique, "wine_id must be unique"

Silver shape: (135192, 24)


## Categorical encoding (ordinal)

Leakage-free label map. `encoded_missing_value=-1` keeps NaNs as their
own code.

In [2]:
cat_cols = ["country", "wine_type", "state", "appellation", "varietal_label", "company"]

enc = OrdinalEncoder(encoded_missing_value=-1)
df[[f"{c}_ord" for c in cat_cols]] = enc.fit_transform(df[cat_cols])

show(df[[c for col in cat_cols for c in (col, f"{col}_ord")]].value_counts().reset_index(), maxBytes="2MB")

Loading ITables v2.7.3 from the internet... (need help?)


## Engineered numerics

- `log_retail` — model on the log of the right-skewed price target.
- `age_at_review` — bottle age when reviewed: `date_of_review` year − `vintage`.
  Non-vintage (NV) wines have no `vintage` (NaN), so their age is left **null**
  rather than fabricated.

In [3]:
df["log_retail"] = np.log(df["retail"])

# Bottle age at review. vintage is NaN for NV wines → age_at_review stays NaN.
review_year = pd.to_datetime(df["date_of_review"], errors="coerce").dt.year
df["age_at_review"] = review_year - df["vintage"]

print(f"age_at_review null: {df['age_at_review'].isna().sum():,} "
      f"(of which NV: {df['is_nv'].sum():,})")
df[["retail", "log_retail", "vintage", "age_at_review"]].describe().T

age_at_review null: 4,618 (of which NV: 4,618)


,count,mean,std,min,25%,50%,75%,max
retail,127160.0,44.497141,73.184722,1.0,20.000000,32.000000,53.000000,9999.990000
log_retail,127160.0,3.529189,0.667833,0.0,2.995732,3.465736,3.970292,9.210339
vintage,130574.0,2019.519889,2.464731,1961.0,2018.000000,2019.000000,2021.000000,2025.000000
age_at_review,130574.0,2.672822,1.794155,-20.0,2.000000,2.000000,3.000000,62.000000


## Assemble & save feature set

`wine_id` + numerics + ordinal codes + target. One row per wine, joins
to any other feature table (keywords, embeddings) on `wine_id`.

In [4]:
numeric_features = ["rating", "alcohol", "bottle_size", "vintage", "case_production", "age_at_review"]
ordinal_features = [f"{c}_ord" for c in cat_cols]
target_cols      = ["retail", "log_retail"]

feature_cols = ["wine_id"] + numeric_features + ordinal_features + target_cols
features_basic = df[feature_cols].copy()

features_basic.to_parquet(FEATURES_PATH, index=False)
print(f"Saved {features_basic.shape[0]:,} rows x {features_basic.shape[1]} cols -> {FEATURES_PATH}")
features_basic.head()

Saved 135,192 rows x 15 cols -> ..\..\.data\features_basic.parquet


,wine_id,rating,alcohol,bottle_size,vintage,case_production,age_at_review,country_ord,wine_type_ord,state_ord,appellation_ord,varietal_label_ord,company_ord,retail,log_retail
0,0,91,0.1420,750.0,2017.0,600.0,2.0,40.0,5.0,4.0,223.0,670.0,8070.0,90.0,4.499810
1,1,93,0.1410,750.0,2017.0,501.0,2.0,40.0,5.0,4.0,1202.0,670.0,8075.0,48.0,3.871201
2,2,92,0.1423,750.0,2017.0,176.0,3.0,40.0,5.0,4.0,1189.0,670.0,3049.0,28.0,3.332205
3,3,88,0.1350,750.0,2017.0,1050.0,3.0,40.0,5.0,4.0,666.0,670.0,6669.0,18.0,2.890372
4,4,88,0.1470,750.0,2017.0,150.0,3.0,40.0,5.0,4.0,1124.0,670.0,8026.0,55.0,4.007333
